# Fine-tuning Pipeline (MACE-MH-1 OMAT)

This notebook uses multi-head tuning with 2 datasets:
    * DFT relaxation set constructed below
    * configs from OMAT head w/ energy + force labels from MACE-MH-1 predictions (to preserve original model behavior)
      replay source [06/23/26]: https://github.com/ACEsuit/mace-foundations/releases/tag/mace_mh_1

See https://mace-docs.readthedocs.io/en/latest/guide/multihead_finetuning.html for more details.

## Conda Env Setup
In the CLI, run 
`path/mace-mh-1.yml`

In [1]:
from ase.io import iread, read, write
from pathlib import Path

In [2]:
# read in list of selected DFT run folder names from cache
foldernames = []

with open("../data-processing/cache/screened_foldernames.txt", "r") as f:
    for line in f:
        line.strip()
        line.removesuffix("-del1")
        foldernames.append(line)

print(f"Total items: {len(foldernames)}")

Total items: 98


In [7]:
from collections import defaultdict
import re

def make_outcar_paths():
    # return a list of paths to the OUTCARs of each VASP run
    outcar_dir = Path("../data-processing/outcars/")
    
    groups = defaultdict(list)  # dict creates empty list for any new key

    for p in outcar_dir.iterdir():
        m = re.match(r"(.+)_OUTCAR(?:-(\d+))?$", p.name)
        if m:
            name, num = m.group(1), m.group(2)
            groups[name].append((int(num) if num else 0, p))

    outcar_paths = []
    for name, files in groups.items():  # sort by name end enumeration
        files.sort(key=lambda x: x[0])
        outcar_paths.append((name, [f[1] for f in files]))

    return outcar_paths

In [8]:
# OUTCARP_PATHS: list of paths to OUTCAR file for each run
OUTCAR_PATHS = make_outcar_paths()

In [12]:
# create finetune set by storing the first and last frames of each DFT run as an ASE Atoms object

def load_frames(paths):
    # return tuple of first and last frames of DFT run
    _, outcar_paths = paths
    
    # grab 5th frame as "first" to let calculations evolve from initial guess
    i = 0
    found = False
    for p in outcar_paths:  # continue to next OUTCAR if first has <5 frames
        for frame in iread(p, format="vasp-out"):
            first = frame
            if i == 4:
                found = True
                break
            i += 1
        if found:
            break
    if not found:
        print(f"< 5 frames total across {paths} (only {i+1})")
                
    last = read(outcar_paths[-1], index=-1, format="vasp-out")

    return first, last

In [ ]:
# test OUTCAR read-in here

test_frame = load_frames(OUTCAR_PATHS[0])
print(test_frame[0].get_total_energy())
print(test_frame[1].get_total_energy())

-120.67749849
-125.14691216


In [19]:
dft_frames_set = []
HEAD_NAME = "dft_ag"

for path in OUTCAR_PATHS:
    try:
        first, last = load_frames(path)
        for atoms in [first, last]:
            # atoms = atoms.get_potential_energy()
            atoms.info["REF_energy"] = atoms.calc.results["energy"]
            atoms.info["head"] = HEAD_NAME
            atoms.arrays["REF_forces"] = atoms.calc.results["forces"]
        dft_frames_set.extend([first, last])
    except Exception as e:
        print(f"FAILED: {path} -> {e}")
        continue

write("../training-data/mace-ft-input.xyz", dft_frames_set, format="extxyz")

FAILED: ('Ag_rec_0_OCCO_more_0', [PosixPath('../data-processing/outcars/Ag_rec_0_OCCO_more_0_OUTCAR-1')]) -> Expected length of ion_types to be same as species, but got ion_types=10 and species=11


Check the .xyz file of training data you just wrote.

In [ ]:
# check frame counts

check = read("../training-data/mace-ft-input.xyz", index=":")
print(f"{len(OUTCAR_PATHS)} paths to DFT runs --> 2 * {len(OUTCAR_PATHS)} - 2 * FAILED = ")
print(len(check), "frames written")
print(check[0].info)          # should show REF_energy, head
print(check[0].arrays.keys()) # should include REF_forces

98 paths to DFT runs --> 2 * 98 - 2 * FAILED = 
194 frames written
{'REF_energy': np.float64(-120.67749849), 'head': 'dft_ag'}
dict_keys(['numbers', 'positions', 'REF_forces'])


In [ ]:
# check head label consistency

set(a.info["head"] for a in check) == {"dft_ag"}

True

**Create the combined finetuning set** with the following CLI command.
Run in the finetuned-mlips/data/ folder.
This calls fine_tuning_select.py (stored in the mace-torch package you installed in your conda env) 
to create a filtered 3rd file from your dft data and the replay. Edit flags below filepaths at will.

See all possible parameters and more info w/ `python -m mace.cli.fine_tuning_select --help`

```bash 
python -m mace.cli.fine_tuning_select \
  --configs_pt data/replay-data-mh-1-omat-pbe.xyz \
  --configs_ft data/train.xyz \
  --num_samples 10000 \
  --subselect fps \
  --model path/to/foundation_model.model \
  --output selected_configs.xyz \
  --filtering_type combinations \
  --head_pt omat_pbe \
  --head_ft tuned_head \
  --weight_pt 1.0 \
  --weight_ft 10.0
```
Parameters:
* `--configs_pt`: Path to the replay dataset
* `--configs_ft`: Path to your target dataset
* `--num_samples`: Number of configurations to select from the replay dataset
* `--subselect`: Method for subselection (fps for Farthest Point Sampling or random)
* `--filtering_type`: How to filter configurations based on elements:
    * `combinations`: Keep configurations with combinations of elements in your target dataset
    * `exclusive`: Keep configurations containing only elements in your target dataset
    * `inclusive`: Keep configurations containing all elements in your target dataset
    * `none`: No filtering
* `--filter_atomic_numbers_pt`: Optionally specify specific atomic numbers to filter by
* `--weight_pt`: weights the loss calculated on replay (pretraining refs) structures
* `--weight_ft`: weights the loss calcualted on finetune structures

Copy/Paste Version:
```bash
python -m mace.cli.fine_tuning_select --configs_pt ../training-data/replay-data-mh-1-omat-pbe.xyz --configs_ft ../training-data/mace-ft-input.xyz --num_samples 811 --subselect fps --model /Users/zschwab/.cache/huggingface/hub/models--mace-foundations--mace-mh-1/snapshots/95fc198fe533fd351b4b74bd0fdbdc6c10a2dddd/mace-mh-1.model --output ../training-data/selected_configs.xyz --filtering_type combinations --filter_atomic_numbers_pt "[1, 6, 8, 47]" --head_pt omat_pbe --head_ft dft_ag --weight_pt 1.0 --weight_ft 10.0 --default_dtype float32 --disallow_random_padding
```

**Get EOs** (energies of single isolated atoms) for reference

In [4]:
# USER INPUT HERE: enter atoms
ATOMS = ["Ag", "C", "H", "O"]

REF_PATH = "../training-data/dft-data/"

def get_eo(atom):
    frame = read(f"{REF_PATH}{atom}_Gas_OUTCAR", index=-1, format="vasp-out")
    energy = frame.get_potential_energy()
    print(f"{atom}: {energy} eV")
    return energy

In [5]:
for a in ATOMS:
    get_eo(a)

Ag: -0.19458509 eV
C: -1.28369203 eV
H: -1.11167391 eV
O: -1.55774442 eV


**Train model on combined set** with the following CLI command/bash script.

```bash
python -m mace.cli.run_train \
--name Ag_CHO_finetuned_mace-mh-1 \
--train_file data/selected_configs.xyz \
--foundation_model mh-1 \
--foundation_head omat_pbe \
--pt_train_file data/selected_configs_combined.xyz \
--energy_weight 1.0 \
--forces_weight 100.0 \
--swa \
--swa_energy_weight 10.0 \
--swa_forces_weight 100.0 \
--stress_weight 0.0 \
--lr 5e-4 \
--multiheads_finetuning True \
--valid_fraction 0.1 \
--E0s "{47: -0.19458509, 8: -1.55774442, 6: -1.28369203, 1: -1.11167391}" \
--max_num_epochs 30 \
--device "cuda" \
```